# Amazon.in Laptop Scraper

This notebook scrapes laptop listings from **Amazon.in** and exports the results to a **timestamped CSV file**.


## 1. Install & Import Dependencies

In [ ]:
# Install required libraries (run once)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                       'requests', 'beautifulsoup4', 'lxml', 'pandas', '--quiet'])
print('✅ Dependencies ready')

In [ ]:
import csv
import time
import random
import logging
from datetime import datetime
from urllib.parse import urlencode, urljoin

import requests
from bs4 import BeautifulSoup
import pandas as pd

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s [%(levelname)s] %(message)s',
                    datefmt='%H:%M:%S')
log = logging.getLogger(__name__)
print('✅ Imports successful')

## 2. Configuration — Edit Here

In [ ]:
# USER SETTINGS 
SEARCH_QUERY = 'laptop'       # Change to 'gaming laptop', 'MacBook', etc.
PAGES_TO_SCRAPE = 3           # Number of Amazon search pages (≈16 products/page)
DELAY_MIN = 3                 # Minimum wait (seconds) between pages
DELAY_MAX = 7                 # Maximum wait (seconds) between pages


BASE_URL   = 'https://www.amazon.in'
SEARCH_URL = f'{BASE_URL}/s'

USER_AGENTS = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
    '(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 '
    '(KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36',
    'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 '
    '(KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:125.0) Gecko/20100101 Firefox/125.0',
]

HEADERS_BASE = {
    'Accept-Language': 'en-IN,en;q=0.9',
    'Accept-Encoding': 'gzip, deflate, br',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Connection': 'keep-alive',
    'DNT': '1',
    'Upgrade-Insecure-Requests': '1',
    'Referer': 'https://www.amazon.in/',
}
print('✅ Configuration set')

## 3. Helper Functions

In [ ]:
def make_session() -> requests.Session:
    """Return a requests session pre-loaded with browser-like headers."""
    session = requests.Session()
    session.headers.update(HEADERS_BASE)
    return session


def fetch_page(session: requests.Session, query: str, page: int):
    """Download one search-results page. Returns BeautifulSoup or None."""
    params = {
        'k': query,
        'i': 'computers',
        'page': page,
        'ref': f'sr_pg_{page}',
    }
    url = f"{SEARCH_URL}?{urlencode(params)}"
    headers = {'User-Agent': random.choice(USER_AGENTS)}

    log.info(f'Fetching page {page}: {url}')
    try:
        resp = session.get(url, headers=headers, timeout=20)
        resp.raise_for_status()

        if 'api-services-support@amazon.com' in resp.text or 'Something went wrong' in resp.text:
            log.warning('⚠️  CAPTCHA / bot-check page detected — stopping.')
            return None

        return BeautifulSoup(resp.text, 'lxml')

    except requests.RequestException as exc:
        log.error(f'Request failed: {exc}')
        return None


def parse_card(card: BeautifulSoup, scraped_at: str) -> dict | None:
    """Extract all fields from a single product card."""

    # Ad vs Organic 
    sponsored_tag = card.select_one(
        'span.puis-sponsored-label-text, '
        'span[data-component-type="s-ads-metrics"] span'
    )
    result_type = 'Ad' if sponsored_tag else 'Organic'

    # Title 
    title_tag = card.select_one('h2 span.a-text-normal, h2 a span')
    title = title_tag.get_text(strip=True) if title_tag else 'N/A'

    # Price 
    price_whole = card.select_one('span.a-price-whole')
    price_frac  = card.select_one('span.a-price-fraction')
    if price_whole:
        whole = price_whole.get_text(strip=True).replace(',', '').replace('.', '')
        frac  = price_frac.get_text(strip=True) if price_frac else '00'
        price = f'₹{whole}.{frac}'
    else:
        price = 'N/A'

    #  Rating 
    rating_tag = card.select_one('span.a-icon-alt')
    if rating_tag:
        rt = rating_tag.get_text(strip=True)
        rating = rt.split(' out of')[0] if 'out of' in rt else rt
    else:
        rating = 'N/A'

    # Reviews 
    reviews_tag = card.select_one('span.a-size-base.s-underline-text, span[aria-label*="ratings"]')
    if reviews_tag:
        reviews = reviews_tag.get_text(strip=True).replace(',', '')
    else:
        reviews_link = card.select_one('a[href*="customerReviews"] span')
        reviews = reviews_link.get_text(strip=True).replace(',', '') if reviews_link else 'N/A'

    # Image URL 
    img_tag = card.select_one('img.s-image')
    image_url = img_tag.get('src', 'N/A') if img_tag else 'N/A'

    # Product URL 
    link_tag = card.select_one('a.a-link-normal.s-no-outline, h2 a')
    if link_tag and link_tag.get('href'):
        product_url = urljoin(BASE_URL, link_tag['href'].split('?')[0])
    else:
        product_url = 'N/A'

    if title == 'N/A' and price == 'N/A' and image_url == 'N/A':
        return None

    return {
        'Title':       title,
        'Price (INR)': price,
        'Rating':      rating,
        'Reviews':     reviews,
        'Result Type': result_type,
        'Image URL':   image_url,
        'Product URL': product_url,
        'Scraped At':  scraped_at,
    }

print('✅ Helper functions defined')

## 4. Run the Scraper

In [ ]:
session    = make_session()
products   = []
scraped_at = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

for page in range(1, PAGES_TO_SCRAPE + 1):
    soup = fetch_page(session, SEARCH_QUERY, page)
    if soup is None:
        print(f'⚠️  Stopping at page {page} — fetch failed or CAPTCHA.')
        break

    cards = soup.select("div[data-component-type='s-search-result']")
    log.info(f'  → {len(cards)} cards found on page {page}')

    for card in cards:
        record = parse_card(card, scraped_at)
        if record:
            products.append(record)

    if page < PAGES_TO_SCRAPE:
        delay = random.uniform(DELAY_MIN, DELAY_MAX)
        log.info(f'  Sleeping {delay:.1f}s …')
        time.sleep(delay)

print(f'\n🎯 Scraping complete — {len(products)} products collected.')

## 5. Save to Timestamped CSV

In [ ]:
if products:
    df = pd.DataFrame(products)

    # Reorder columns
    col_order = ['Title', 'Price (INR)', 'Rating', 'Reviews',
                 'Result Type', 'Image URL', 'Product URL', 'Scraped At']
    df = df[[c for c in col_order if c in df.columns]]

    #Timestamped filename 
    timestamp   = datetime.now().strftime('%Y%m%d_%H%M%S')
    safe_query  = SEARCH_QUERY.replace(' ', '_')
    output_file = f'amazon_{safe_query}_{timestamp}.csv'

    df.to_csv(output_file, index=False, encoding='utf-8-sig', quoting=csv.QUOTE_ALL)
    print(f'✅ Saved {len(df)} rows → "{output_file}"')
else:
    print('⚠️  No data collected — nothing to save.')
    output_file = None

## 6. Preview the Results

In [ ]:
if products:
    pd.set_option('display.max_colwidth', 55)
    pd.set_option('display.max_columns', None)
    print(f'Shape: {df.shape[0]} rows × {df.shape[1]} columns\n')
    df.head(10)

## 7. Quick Analysis

In [ ]:
if products:
    print('=== Ad vs Organic breakdown ===')
    print(df['Result Type'].value_counts().to_string())

    print('\n=== Rating distribution (top values) ===')
    print(df['Rating'].value_counts().head(10).to_string())

    print('\n=== Price range ===')
    prices_num = (
        df['Price (INR)']
        .str.replace('[₹,]', '', regex=True)
        .str.split('.').str[0]
        .pipe(pd.to_numeric, errors='coerce')
    )
    print(prices_num.describe().to_string())